# Lesson 16 — Memory, Buffer Pool & the Log

> **Prerequisite:** Tiers 1–4. Pairs with lesson 13 (plans) for the spill discussion.

## What you'll learn
- The buffer pool (data cache): how data lives in memory, clean vs dirty pages
- Checkpoint and the lazy writer; Page Life Expectancy as a memory-pressure signal
- Write-ahead logging (WAL), the transaction log, and Virtual Log Files (VLFs)
- Why the log won't truncate (`log_reuse_wait_desc`)
- Memory grants and tempdb spills

## Setup
Run `setup.sql` once. It creates the `lesson16` schema with `BigDetail` — an inflated copy of `SalesOrderDetail` large enough to make caching and spills visible. **First run may take a minute or two** to build the table. Re-run to reset.

**Tooling note:** run the SQL cells in this notebook with Azure Data Studio connected to the mssql-learn container (attach the SQL kernel to your saved localhost connection). `CHECKPOINT` and `DBCC DROPCLEANBUFFERS` are dev-only diagnostics — they are safe on this learning container but must never be run on a shared/production server. Example 7 uses the actual execution plan (Ctrl+M, an SSMS feature) to see the Sort's "spilled to tempdb" warning, and returns ~485k rows — that is expected.

In [ ]:
USE AdventureWorks2022;
GO

## Concepts

### The buffer pool
SQL Server reads/writes 8KB pages in memory, not on disk directly. The **buffer pool** is that data cache. A page modified in memory is **dirty** until a **checkpoint** (or the **lazy writer** under memory pressure) flushes it to disk. `sys.dm_os_buffer_descriptors` shows what is cached.

### Page Life Expectancy (PLE)
The expected seconds a page stays cached before eviction. A healthy busy server keeps PLE high and stable; a sharp drop means SQL Server is churning the cache (memory pressure). Read it from `sys.dm_os_performance_counters`. On an idle dev box a low PLE can be meaningless — watch the *trend*.

### Write-ahead logging & the log
Before a data page change is written, the change is written to the **transaction log** (WAL). The log is divided into **VLFs** (`sys.dm_db_log_info`). The log can only be truncated (reused) once a portion is no longer needed; `sys.databases.log_reuse_wait_desc` tells you what it's waiting on — most commonly `ACTIVE_TRANSACTION` (an open transaction) or a pending backup in FULL recovery.

### Memory grants & spills
Sorts and hash joins request a **memory grant** before executing, sized from cardinality estimates. If the grant is too small (bad estimate, or forced), the operator **spills to tempdb** — extra slow I/O flagged on the plan operator. `sys.dm_exec_query_memory_grants` shows grants.

## Worked investigations

### Example 1 — How many pages of each database are currently cached?

Cached pages per database.

In [ ]:
SELECT
    CASE database_id WHEN 32767 THEN 'ResourceDB'
         ELSE DB_NAME(database_id) END AS DatabaseName,
    COUNT(*)                            AS CachedPages,
    COUNT(*) * 8 / 1024.0               AS CachedMB
FROM sys.dm_os_buffer_descriptors
GROUP BY database_id
ORDER BY CachedPages DESC;

### Example 2 — Show a table being pulled into cache

A table loading into cache before/after a scan.

In [ ]:
-- Clear THIS database's cached pages first (dev only — never in production)
-- DBCC DROPCLEANBUFFERS clears clean pages server-wide; CHECKPOINT flushes dirty ones first.
CHECKPOINT;
DBCC DROPCLEANBUFFERS;

-- How much of BigDetail is cached now? (expect ~0)
SELECT COUNT(*) AS BigDetailPagesCached
FROM sys.dm_os_buffer_descriptors AS bd
WHERE bd.database_id = DB_ID()
  AND bd.allocation_unit_id IN (
        SELECT au.allocation_unit_id
        FROM sys.allocation_units AS au
        JOIN sys.partitions AS p ON p.partition_id = au.container_id
        WHERE p.object_id = OBJECT_ID('lesson16.BigDetail'));

-- Scan the table to load it into the buffer pool
SELECT COUNT(*) FROM lesson16.BigDetail;

-- Now many more pages are cached
SELECT COUNT(*) AS BigDetailPagesCached
FROM sys.dm_os_buffer_descriptors AS bd
WHERE bd.database_id = DB_ID()
  AND bd.allocation_unit_id IN (
        SELECT au.allocation_unit_id
        FROM sys.allocation_units AS au
        JOIN sys.partitions AS p ON p.partition_id = au.container_id
        WHERE p.object_id = OBJECT_ID('lesson16.BigDetail'));

### Example 3 — Page Life Expectancy (seconds a page is expected to stay in cache)

Page Life Expectancy.

In [ ]:
-- Higher is better; a sharply falling PLE signals memory pressure.
SELECT object_name, counter_name, cntr_value AS PLE_Seconds
FROM sys.dm_os_performance_counters
WHERE counter_name = 'Page life expectancy'
  AND object_name LIKE '%Buffer Manager%';

### Example 4 — Inspect the Virtual Log Files (VLFs) of the current database

VLF count.

In [ ]:
SELECT COUNT(*) AS VLF_Count
FROM sys.dm_db_log_info(DB_ID());

### Example 5 — Why can't the log be truncated right now?

`log_reuse_wait_desc`.

In [ ]:
-- log_reuse_wait_desc tells you what the log is waiting on (NOTHING, ACTIVE_TRANSACTION, etc.)
SELECT name, log_reuse_wait_desc, recovery_model_desc
FROM sys.databases
WHERE name = 'AdventureWorks2022';

### Example 6 — An open transaction holds the log hostage

An open transaction holding the log (ACTIVE_TRANSACTION).

In [ ]:
BEGIN TRANSACTION;
UPDATE lesson16.BigDetail SET OrderQty = OrderQty WHERE SalesOrderDetailID % 1000 = 0;

-- While the transaction is OPEN, check log reuse wait again — expect ACTIVE_TRANSACTION
SELECT name, log_reuse_wait_desc FROM sys.databases WHERE name = 'AdventureWorks2022';

-- See the open transaction
SELECT
    DB_NAME(dt.database_id)   AS DatabaseName,
    st.session_id,
    at.name                   AS TranName,
    at.transaction_begin_time
FROM sys.dm_tran_active_transactions  AS at
JOIN sys.dm_tran_session_transactions AS st ON st.transaction_id = at.transaction_id
JOIN sys.dm_tran_database_transactions AS dt ON dt.transaction_id = at.transaction_id
WHERE dt.database_id = DB_ID();

COMMIT TRANSACTION;   -- release it; log_reuse_wait_desc returns to NOTHING

### Example 7 — Force a tempdb spill by starving the sort of memory

A forced tempdb spill + memory-grant DMV.

In [ ]:
-- A big sort needs a memory grant. MAX_GRANT_PERCENT caps it tiny, forcing a spill to tempdb.
-- Run with the ACTUAL plan (Ctrl+M): the Sort operator shows a warning triangle "spilled to tempdb".
-- (Returns ~485k rows — that is expected.)
SELECT *
FROM lesson16.BigDetail
ORDER BY ProductID, ModifiedDate, OrderQty, UnitPrice
OPTION (MAX_GRANT_PERCENT = 0.0);   -- starve the grant to guarantee a spill

-- Inspect active/last memory grants
SELECT TOP 5
    mg.session_id,
    mg.requested_memory_kb,
    mg.granted_memory_kb,
    mg.required_memory_kb,
    mg.dop
FROM sys.dm_exec_query_memory_grants AS mg;

## Common issues this explains
- Low PLE / memory pressure on busy systems.
- A transaction log that grows uncontrollably because of a long-open transaction.
- tempdb spills slowing big sorts/joins.

## Pitfalls
- `DBCC DROPCLEANBUFFERS` / `CHECKPOINT` are dev-only diagnostics — never run them to "fix" production.
- Many tiny VLFs (log autogrown in small increments) slow recovery and log operations.
- A single forgotten open transaction can stop log truncation for the whole database.
- Memory-grant numbers vary by server memory and DOP — compare direction, not absolute KB.

## Cheatsheet
See `cheatsheets/06-execution-plans.md`

## Exercises
Open `exercises.ipynb` and work them in order. Solutions in `exercises-solutions.ipynb`.